In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


def add_three(state: State):

    print("ADD_THREE running...")

    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)
builder.add_node("add_three", add_three)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", "add_three")
builder.add_edge("add_three", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. THREAD
# =========================================================

config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


# =========================================================
# 6. ORIGINAL EXECUTION
# =========================================================

print("\n========================================")
print("ORIGINAL EXECUTION")
print("========================================")

result_1 = graph.invoke(
    {
        "count": 0
    },
    config=config
)

print("\nOriginal Result:")
print(result_1)


# =========================================================
# 7. GET HISTORY
# =========================================================

history = list(
    graph.get_state_history(config)
)

print("\n========================================")
print("CHECKPOINT HISTORY")
print("========================================")

for index, state in enumerate(history):

    print(f"\nCheckpoint {index}")

    print(
        "Checkpoint ID:",
        state.config["configurable"]["checkpoint_id"]
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 8. FIND A CHECKPOINT WITH A NEXT NODE
# =========================================================

target_state = None

for state in history:

    if state.next:

        target_state = state
        break


if target_state is None:

    print(
        "\nNo historical checkpoint with a next node found."
    )

else:

    # =====================================================
    # 9. OLD CHECKPOINT
    # =====================================================

    old_checkpoint_id = (
        target_state
        .config["configurable"]["checkpoint_id"]
    )

    old_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": old_checkpoint_id
        }
    }

    old_state = graph.get_state(
        old_config
    )

    print("\n========================================")
    print("SELECTED OLD CHECKPOINT")
    print("========================================")

    print(
        "Checkpoint ID:",
        old_checkpoint_id
    )

    print(
        "Old Values:",
        old_state.values
    )

    print(
        "Next Node:",
        old_state.next
    )


    # =====================================================
    # 10. UPDATE HISTORICAL STATE
    # =====================================================

    print("\n========================================")
    print("UPDATE STATE")
    print("========================================")

    updated_config = graph.update_state(
        old_config,
        {
            "count": 100
        }
    )

    print(
        "Updated Config:"
    )

    print(
        updated_config
    )


    # =====================================================
    # 11. READ UPDATED STATE
    # =====================================================

    updated_state = graph.get_state(
        updated_config
    )

    print("\n========================================")
    print("UPDATED STATE")
    print("========================================")

    print(
        "Thread ID:",
        updated_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "New Checkpoint ID:",
        updated_state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        "Values:",
        updated_state.values
    )

    print(
        "Next Node:",
        updated_state.next
    )


    # =====================================================
    # 12. CONTINUE FROM UPDATED STATE
    # =====================================================

    print("\n========================================")
    print("EXECUTION FROM UPDATED STATE")
    print("========================================")

    result_2 = graph.invoke(
        None,
        config=updated_config
    )

    print(
        "\nNew Execution Result:"
    )

    print(
        result_2
    )


    # =====================================================
    # 13. FINAL STATE
    # =====================================================

    final_state = graph.get_state(
        updated_config
    )

    print("\n========================================")
    print("FINAL STATE")
    print("========================================")

    print(
        "Thread ID:",
        final_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Checkpoint ID:",
        final_state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        "Values:",
        final_state.values
    )

    print(
        "Next:",
        final_state.next
    )


    # =====================================================
    # 14. FINAL HISTORY
    # =====================================================

    print("\n========================================")
    print("HISTORY AFTER UPDATE")
    print("========================================")

    new_history = list(
        graph.get_state_history(config)
    )

    for index, state in enumerate(new_history):

        print(f"\nCheckpoint {index}")

        print(
            "ID:",
            state.config[
                "configurable"
            ]["checkpoint_id"]
        )

        print(
            "Values:",
            state.values
        )

        print(
            "Next:",
            state.next
        )